# 01 — Dataset Exploration

Understand what data we have **before training**:
- Detector dataset split sizes and class distribution
- Bounding box size distribution (validates severity thresholds)
- Severity crop distribution (Low / Medium / High)
- Sample annotated images from each class

In [ ]:
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import Counter
import torch

BASE_DIR = Path('..').resolve()
print('Project root:', BASE_DIR)

# GPU info
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('GPU: not available')

try:
    import psutil
    print(f'RAM: {psutil.virtual_memory().total/1e9:.1f} GB')
except ImportError:
    print('RAM: install psutil for RAM info (pip install psutil)')

## 1. Detector Dataset — Split Sizes

In [ ]:
DETECTOR = BASE_DIR / 'data' / 'processed' / 'detector_yolo'
DETECTOR_AUG = BASE_DIR / 'data' / 'processed' / 'detector_yolo_aug'

assert DETECTOR.exists(), 'Run python src/prepare_data.py first'

print('  Detector dataset (detector_yolo/)')
for split in ['train', 'val', 'test']:
    imgs = list((DETECTOR / 'images' / split).glob('*'))
    lbls = list((DETECTOR / 'labels' / split).glob('*.txt'))
    print(f'    {split:5s} → {len(imgs):5d} images  {len(lbls):5d} label files')

if DETECTOR_AUG.exists():
    aug_imgs = list((DETECTOR_AUG / 'images' / 'train').glob('*'))
    print(f'\n  Augmented train split (detector_yolo_aug/):')
    print(f'    train → {len(aug_imgs):5d} images')

## 2. Class Distribution per Split

In [ ]:
names = {0: 'pothole', 1: 'traffic_light'}
colors_cls = {0: '#e74c3c', 1: '#2ecc71'}

split_counts = {}
for split in ['train', 'val', 'test']:
    counts = Counter()
    for lbl_path in (DETECTOR / 'labels' / split).glob('*.txt'):
        for line in lbl_path.read_text().strip().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    split_counts[split] = counts

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, split in zip(axes, ['train', 'val', 'test']):
    counts = split_counts[split]
    labels = [names[k] for k in sorted(counts)]
    vals   = [counts[k] for k in sorted(counts)]
    clrs   = [colors_cls[k] for k in sorted(counts)]
    bars   = ax.bar(labels, vals, color=clrs)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(v), ha='center', fontsize=10, fontweight='bold')
    ax.set_title(f'{split.capitalize()} split', fontsize=12)
    ax.set_ylabel('Boxes')

plt.suptitle('Bounding Box Count per Class per Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Bounding Box Size Distribution

Validates severity auto-label thresholds:
- `< 3%` of image area → **Low**
- `3–12%` → **Medium**
- `≥ 12%` → **High**

In [ ]:
pothole_ratios = []

for lbl_path in (DETECTOR / 'labels' / 'train').glob('*.txt'):
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) == 5 and int(parts[0]) == 0:   # class 0 = pothole
            bw, bh = float(parts[3]), float(parts[4])
            pothole_ratios.append(bw * bh)

print(f'Pothole boxes analysed: {len(pothole_ratios)}')

low    = sum(1 for r in pothole_ratios if r < 0.03)
medium = sum(1 for r in pothole_ratios if 0.03 <= r < 0.12)
high   = sum(1 for r in pothole_ratios if r >= 0.12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(pothole_ratios, bins=60, color='#e74c3c', edgecolor='white', alpha=0.85)
axes[0].axvline(0.03, color='#f39c12', linestyle='--', linewidth=2, label='Low/Med (0.03)')
axes[0].axvline(0.12, color='#c0392b', linestyle='--', linewidth=2, label='Med/High (0.12)')
axes[0].set_title('Pothole Box Area Distribution (train)')
axes[0].set_xlabel('box_w × box_h  (normalised)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].pie(
    [low, medium, high],
    labels=[f'Low\n{low}', f'Medium\n{medium}', f'High\n{high}'],
    autopct='%1.1f%%',
    colors=['#27ae60', '#f39c12', '#e74c3c'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
axes[1].set_title('Auto-labelled Severity Distribution')

plt.tight_layout()
plt.show()
print(f'Low: {low}  Medium: {medium}  High: {high}  (total: {low+medium+high})')

## 4. Severity Crop Counts

In [ ]:
SEVERITY = BASE_DIR / 'data' / 'processed' / 'severity_crops'
assert SEVERITY.exists(), 'Run python src/prepare_data.py first'

classes = ['Low', 'Medium', 'High']
x       = np.arange(len(classes))
width   = 0.25

fig, ax = plt.subplots(figsize=(8, 4))
split_clrs = ['#3498db', '#e67e22', '#2ecc71']
for i, (split, clr) in enumerate(zip(['train', 'val', 'test'], split_clrs)):
    counts = [len(list((SEVERITY / split / cls).glob('*')))
               if (SEVERITY / split / cls).exists() else 0
               for cls in classes]
    bars = ax.bar(x + i * width, counts, width, label=split, color=clr, alpha=0.85)
    for bar, v in zip(bars, counts):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    str(v), ha='center', fontsize=8)

ax.set_xticks(x + width); ax.set_xticklabels(classes)
ax.set_title('Severity Crop Counts per Split')
ax.set_ylabel('Number of crops')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Sample Detector Images with Annotations

In [ ]:
def show_yolo_samples(images_dir, labels_dir, n=6, title=''):
    img_paths = sorted(images_dir.glob('*'))[:n]
    if not img_paths:
        print(f'No images found in {images_dir}'); return

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    cls_colors = {0: (1.0, 0.3, 0.3), 1: (0.3, 0.9, 0.3)}

    for ax, img_path in zip(axes.flatten(), img_paths):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl  = labels_dir / (img_path.stem + '.txt')
        if lbl.exists():
            for line in lbl.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) != 5: continue
                cls, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)
                c  = cls_colors.get(cls, (0.8, 0.8, 0.8))
                ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                             linewidth=2, edgecolor=c, facecolor='none'))
                label_name = 'pothole' if cls == 0 else 'traffic_light'
                ax.text(x1, max(y1-4, 0), label_name, color=c,
                        fontsize=7, fontweight='bold')
        ax.imshow(img); ax.axis('off')
        ax.set_title(img_path.name[:30], fontsize=7)

    plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

show_yolo_samples(
    DETECTOR / 'images' / 'train',
    DETECTOR / 'labels' / 'train',
    title='Sample Training Images — Detector Dataset (pothole=red, traffic_light=green)'
)

## 6. Sample Severity Crops

In [ ]:
sev_colors = {'Low': '#27ae60', 'Medium': '#f39c12', 'High': '#e74c3c'}
fig, axes  = plt.subplots(3, 6, figsize=(15, 8))

for row, sev in enumerate(['Low', 'Medium', 'High']):
    crops = sorted((SEVERITY / 'train' / sev).glob('*'))[:6]
    for col in range(6):
        ax = axes[row][col]
        if col < len(crops):
            img = cv2.cvtColor(cv2.imread(str(crops[col])), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(sev, color=sev_colors[sev], fontsize=9, fontweight='bold')
        ax.axis('off')

plt.suptitle('Severity Crop Samples (Low / Medium / High) — training set', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Dataset Summary

In [ ]:
print('=' * 50)
print('  DATASET SUMMARY')
print('=' * 50)
total_det = 0
for split in ['train', 'val', 'test']:
    n = len(list((DETECTOR / 'images' / split).glob('*')))
    total_det += n
    print(f'  Detector {split:5s}: {n:5d} images')
print(f'  Detector total : {total_det:5d} images')
print()
total_sev = 0
for split in ['train', 'val', 'test']:
    for cls in ['Low', 'Medium', 'High']:
        d = SEVERITY / split / cls
        n = len(list(d.glob('*'))) if d.exists() else 0
        total_sev += n
        print(f'  Severity {split:5s}/{cls:6s}: {n:4d} crops')
print(f'  Severity total : {total_sev:4d} crops')
print()
print('  Ready to train → run 02-train-detector.ipynb')